# Multivariate Statistical Arbitrage on Baskets (VECM / Box–Tiao + GARCH + HMM)

Companion notebook to the Master's thesis *High-Dimensional Statistical Arbitrage: An Integrated KF-HMM-GARCH Framework for Dynamic Multi-Asset Trading* (University of Bologna, 2025/2026). It implements the high-dimensional branch of the empirical part (Part IV, Section 13).

Pairs trading works on two assets; here the mean-reverting spread is built from a **basket** of 3–4 cointegrated assets. The pipeline reuses the machinery of the pairs notebook (GARCH for conditional volatility, HMM for regimes, the same anti-look-ahead discipline) and changes only how the spread is constructed.

**Spread construction** (`Config.basket_method`):

| Method | Spread | Notes |
|---|---|---|
| **VECM** (default) | first Johansen cointegrating vector, $\beta' y_t$ | also returns the loadings $\alpha$ (speeds of adjustment) |
| **Box–Tiao** | most mean-reverting combination (eigenvector of the smallest eigenvalue of a VAR(1) predictability problem) | used automatically as a fallback when the VECM fails |

**Three nested specifications**, mirroring `ols → kf → full` in the pairs branch:

| Spec | Weights | z-score denominator | Regime gate |
|---|---|---|---|
| `raw`    | fixed (estimated on formation) | rolling sample std | none |
| `static` | fixed | GARCH conditional variance | none |
| `full`   | fixed | GARCH conditional variance | HMM, trade only if $\phi_t > \phi^*$ |

`raw → static` isolates the GARCH denominator and `static → full` isolates the HMM gate.

> **Scope limitation.** Unlike the pairs branch, the basket weights stay **fixed** after formation in all three specifications: there is no multivariate Kalman Filter on the weights in this implementation. The comparison therefore isolates GARCH and HMM, not a dynamic hedge ratio (which in the pairs branch is isolated by the `ols → kf` step).

$$\text{candidate baskets} \to \text{VECM / Box–Tiao} \to \text{GARCH} \to \text{HMM} \to \text{signals} \to \text{thresholds} \to \text{backtest}$$

**Out-of-sample discipline.** Weights and models are estimated on the formation window only and applied causally. Thresholds are not optimized on the baskets: they are inherited from the pairs branch (`raw ← ols`, `static`/`full` ← `kf`), which avoids overfitting a handful of baskets. N-leg P&L: $r_t = \tau_{t-1}\sum_i w_i \Delta y_{i,t}$.

### Data and dependencies
The notebook reads the same HDF5 file as the pairs notebook (not distributed with this repository):

| Key | Content |
|---|---|
| `/stocks/prices/adjusted`, `/etfs/prices/adjusted`, `/international/prices/adjusted` | daily adjusted prices, MultiIndex `(date, ticker)`, column `close` |
| `selection/baskets` | candidate baskets: columns `window` (`"<formation start>_<trading end>"`), `group`, `legs` (dict or dict repr keyed by ticker), optionally `rank` |
| `params/thresholds`, `crisis2008_dev/params/thresholds` | thresholds written by the pairs notebook |

`selection/baskets` is produced by an upstream screening step (Johansen tests within sector / asset-class groups) that is not part of this notebook. **Run the pairs notebook first** on the same file, including its 2008–2009 experiment, so that the thresholds exist.

### How to run
Run the cells top to bottom. Sections 1–11 define functions; Section 12 runs the experiments. The pipeline works on a local copy of the HDF5 file and syncs it back to Drive explicitly after each expensive stage (`sync_to_drive`), because many small writes to a FUSE-mounted Drive file can corrupt it.

## 0 · Setup
Mount Google Drive and install the dependencies (`statsmodels` for the VECM, `arch` for GARCH). Outside Colab, skip the first two lines and use `requirements.txt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install arch statsmodels -q

In [ ]:
from __future__ import annotations
import os, ast, shutil, warnings, itertools; warnings.filterwarnings('ignore')
from dataclasses import dataclass
import numpy as np
import pandas as pd
import tables
from scipy.optimize import minimize as _minimize

ANN = np.sqrt(252)   # annualization factor for daily Sharpe ratios

### 0b · Plot style
Same style as the pairs notebook. Colour convention: raw = near-black, static = blue, full = red.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    style='whitegrid', context='paper', font='serif',
    rc={
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'grid.linestyle':    '--',
        'grid.alpha':        0.4,
        'axes.titlesize':    11,
        'axes.labelsize':    10,
        'xtick.labelsize':   8.5,
        'ytick.labelsize':   8.5,
        'legend.fontsize':   8.5,
        'figure.dpi':        150,
        'savefig.dpi':       300,
        'savefig.bbox':      'tight',
    }
)
plt.rcParams['axes.titleweight'] = 'bold'

PALETTE = sns.color_palette('deep', 6)
# Colour map; keys are kept for backward compatibility with the plotting code.
C = {
    'bull':   PALETTE[0],            # blue   -> static
    'gfc':    PALETTE[3],            # red    -> full
    'normal': (0.15, 0.15, 0.15),    # near-black -> raw
}
OUTPUT_DIR = './'   # where figures and CSV tables are saved

## 1 · Configuration
All parameters in one `Config`. Key choices: `basket_method` (`vecm` or `boxtiao`), the half-life range used as a tradability filter, and the fixed regime threshold `phi_star`. `work_local=True` makes every write go to a local copy of the database.

In [ ]:
@dataclass
class Config:
    """All pipeline parameters. Defaults reproduce the thesis setup."""
    hdf_file: str = '/content/drive/MyDrive/data_ip_2026_v2.h5'
    # Work on a local copy and sync to Drive explicitly with sync_to_drive(cfg):
    # many small writes on a FUSE-mounted Drive file can corrupt it.
    work_local: bool = True
    local_path: str = '/content/data_ip_2026_v2_local.h5'

    formation_years: float = 3.0
    embargo_days: int = 10
    dev_end: str = '2019-12-31'
    holdout_start: str = '2020-01-01'
    holdout_end: str = None   # optional upper bound for a bounded holdout (e.g. the 2008-2009 crisis)

    # basket spread construction
    basket_method: str = 'vecm'      # 'vecm' (Johansen VECM, rank 1) | 'boxtiao'
    bt_var_lags: int = 1             # VAR lag for Box-Tiao (VAR(1))
    vecm_k_ar_diff: int = 1          # lagged differences in the VECM (k_ar_diff)
    vecm_deterministic: str = 'ci'   # 'ci' = constant inside the cointegration relation
    half_life_min: float = 5.0
    half_life_max: float = 150.0
    half_life_target: float = 25.0
    min_legs: int = 3

    # spreads (GARCH)
    kf_burnin: int = 20              # initial observations skipped when fitting GARCH
    garch_scale: float = 1e4
    garch_vols: tuple = ('GARCH', 'GJR', 'EGARCH')
    garch_dists: tuple = ('normal', 't', 'skewt')

    # regimes (HMM)
    hmm_restarts: int = 6
    hmm_iters: int = 80
    rv_window: int = 21
    var_floor: float = 1e-4

    # signals
    std_win: int = 252
    std_minp: int = 60

    # thresholds / strategy
    z_stop: float = 4.0
    max_hold_mult: int = 5
    phi_star: float = 0.65
    grid_zstar: tuple = (1.5, 2.0, 2.5, 3.0, 3.5)
    grid_zexit: tuple = (0.25, 0.5, 0.75, 1.0)
    cost_bps: float = 5.0
    cscv_blocks: int = 8

    top_n: int = 10
    seed: int = 0

## 2 · I/O
`Store` reads the prices (read-only) and writes each stage to its own key; `read_base` reads the shared, non-namespaced inputs (prices, `selection/baskets`). The namespace `ns` isolates holdout outputs.

`_h5_is_healthy` checks that every node of the file can be read. `sync_to_drive` copies the local file back to Drive only if that check passes, through a temporary file and an atomic rename. Call it at the end of each expensive stage, not after every write.

In [ ]:
def _h5_is_healthy(path, verbose=False):
    """True if every leaf node of the HDF5 file can be read."""
    try:
        with tables.open_file(path, mode='r') as h5file:
            for node in h5file.walk_nodes('/', classname='Leaf'):
                node.read()
        return True
    except Exception as e:
        if verbose: print(f'[integrity] {path} is NOT readable: {e}')
        return False


def sync_to_drive(cfg: Config, verify=True):
    """Copy the local working file back to Drive (call after each expensive stage).

    With verify=True the copy is aborted, and Drive left untouched, if the local
    file fails the integrity check."""
    if not cfg.work_local:
        print('[sync] work_local=False: writes already go to Drive, nothing to sync.')
        return
    local = cfg.local_path
    if verify and not _h5_is_healthy(local, verbose=True):
        raise RuntimeError(f"[sync] ABORTED: {local} failed the integrity check. "
                            f"Drive was NOT modified.")
    tmp = cfg.hdf_file + '.tmp'
    shutil.copy(local, tmp)
    os.replace(tmp, cfg.hdf_file)
    print(f"[sync] {local} -> {cfg.hdf_file} (verificato: {verify})")


class Store:
    """Thin wrapper around the HDF5 file (prices in, stage outputs out)."""
    def __init__(self, cfg: Config, ns=None):
        self.cfg = cfg; self.ns = ns
        if cfg.work_local:
            self.path = cfg.local_path
            if not os.path.exists(self.path):
                print(f'Copying DB -> {self.path}'); shutil.copy(cfg.hdf_file, self.path)
                if not _h5_is_healthy(self.path, verbose=True):
                    print(f'[WARNING] {self.path} is not readable right after copying it from '
                          f'{cfg.hdf_file}: the file on Drive was already corrupted. '
                          f'Check it before continuing.')
        else:
            self.path = cfg.hdf_file

    def _k(self, key): return f"{self.ns}/{key.strip('/')}" if self.ns else key
    def keys(self):
        with pd.HDFStore(self.path, 'r') as s: return list(s.keys())
    def has(self, key): return ('/'+self._k(key).strip('/')) in self.keys()
    def read(self, key):
        with pd.HDFStore(self.path, 'r') as s: return s[self._k(key)]
    def read_base(self, key):
        """Read WITHOUT namespace: shared inputs (prices, selection/baskets) exist once."""
        with pd.HDFStore(self.path, 'r') as s: return s[key]
    def write(self, key, df, data_columns=None):
        """Write a DataFrame to a (namespaced) key.

        Writing a zero-row DataFrame with format='table' fails silently in PyTables:
        no exception, but the key is never created and a later read raises KeyError.
        Empty frames are therefore stored with format='fixed'."""
        k = self._k(key)
        with pd.HDFStore(self.path, 'a') as s:
            if k in s: s.remove(k)
            if len(df) == 0:
                s.put(k, df, format='fixed')
            else:
                s.put(k, df, format='table', data_columns=data_columns)

    def load_close(self, tickers=None):
        parts = []
        with pd.HDFStore(self.path, 'r') as s:
            keys = set(s.keys())
            for key in ['/stocks/prices/adjusted', '/etfs/prices/adjusted',
                        '/international/prices/adjusted']:
                if key in keys:
                    p = s[key]
                    if tickers is not None:
                        p = p.loc[p.index.get_level_values('ticker').isin(set(tickers))]
                    if len(p): parts.append(p['close'].unstack('ticker'))
        close = pd.concat(parts, axis=1).sort_index()
        return close.loc[:, ~close.columns.duplicated()]

## 3 · Box–Tiao: the most mean-reverting combination
Box & Tiao (1977): fit a VAR(1) on demeaned levels, $y_t = c + \Phi y_{t-1} + e_t$. The *predictability* of a combination $w'y$ is the share of its variance explained by its own past; minimizing it maximizes the speed of mean reversion. The solution is the eigenvector of the **smallest eigenvalue** of the generalized problem

$$(\Phi^\top \Gamma_0 \Phi)\, w = \lambda\, \Gamma_0\, w, \qquad \Gamma_0 = \mathrm{Cov}(y).$$

Weights are normalized to unit L1 norm with a canonical sign (largest weight positive). The half-life is recomputed on the resulting spread.

In [ ]:
def _fit_var1(Y):
    """OLS VAR(1) on levels. Y: (T, N). Returns Phi (N, N)."""
    T, N = Y.shape
    X = Y[:-1]; Z = Y[1:]
    X1 = np.column_stack([np.ones(T-1), X])           # with intercept
    B, *_ = np.linalg.lstsq(X1, Z, rcond=None)        # (N+1, N)
    Phi = B[1:].T                                     # (N, N): Z = c + Phi X
    return Phi

def box_tiao_weights(Y):
    """Y: (T, N) formation log-prices. Returns w (N,), L1-normalized, canonical sign."""
    Y = np.asarray(Y, float)
    Y = Y - Y.mean(0, keepdims=True)                  # demeaned levels
    Gamma0 = np.cov(Y, rowvar=False)                  # (N,N)
    Phi = _fit_var1(Y)                                # (N,N)
    M = Phi.T @ Gamma0 @ Phi                          # numerator of the predictability ratio
    # generalized problem M w = lambda Gamma0 w  -> eigen-decompose Gamma0^{-1} M
    try:
        G0inv = np.linalg.pinv(Gamma0)
        evals, evecs = np.linalg.eig(G0inv @ M)
    except Exception:
        return None
    evals = np.real(evals); evecs = np.real(evecs)
    order = np.argsort(evals)                         # smallest eigenvalue = most mean-reverting
    w = evecs[:, order[0]]
    if not np.all(np.isfinite(w)) or np.allclose(w, 0): return None
    w = w / (np.sum(np.abs(w)) + 1e-12)               # unit L1 norm
    if w[np.argmax(np.abs(w))] < 0: w = -w            # canonical sign: dominant weight > 0
    return w

def _fit_ar1_phi(s):
    s = np.asarray(s, float); x, y = s[:-1], s[1:]
    X = np.column_stack([np.ones_like(x), x]); b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return b[1]

def half_life_of(spread):
    """AR(1) half-life of a spread in days (inf if not mean-reverting)."""
    phi = _fit_ar1_phi(spread)
    if not (0 < phi < 1): return np.inf
    return float(np.log(2)/(-np.log(phi)))

## 4 · VECM (Johansen)
The VECM models the differences as a function of the error-correction term:

$$\Delta y_t = \alpha\,\beta' y_{t-1} + \sum_{i=1}^{k}\Gamma_i \Delta y_{t-i} + \varepsilon_t,$$

where $\beta$ holds the cointegrating vectors (long-run equilibria) and $\alpha$ the loadings (how fast each leg corrects towards equilibrium). With rank $r = 1$ the tradable spread is $\beta_1' y_t$. The loadings are an interpretable output that Box–Tiao does not provide: they show which asset drives the return to equilibrium.

`basket_weights` dispatches between the two methods and falls back to Box–Tiao when the VECM fails to fit.

In [ ]:
def vecm_weights(Y, cfg):
    """Y: (T, N) formation log-prices. Returns (w, alpha_loadings) or (None, None)."""
    try:
        from statsmodels.tsa.vector_ar.vecm import VECM
    except Exception:
        return None, None
    Y = np.asarray(Y, float)
    if Y.shape[0] < 60 or Y.shape[1] < 2: return None, None
    try:
        model = VECM(Y, k_ar_diff=cfg.vecm_k_ar_diff, coint_rank=1,
                     deterministic=cfg.vecm_deterministic)
        res = model.fit()
        beta = np.asarray(res.beta).ravel()[:Y.shape[1]]   # first cointegrating vector
        alpha = np.asarray(res.alpha).ravel()[:Y.shape[1]] # loading
    except Exception:
        return None, None
    if not np.all(np.isfinite(beta)) or np.allclose(beta, 0): return None, None
    # Normalization: beta -> beta/k (unit L1 norm) for trading. alpha and beta are
    # only identified through the product alpha*beta', so alpha is rescaled the
    # opposite way (alpha -> alpha*k) to keep the loadings interpretable as
    # speeds of adjustment.
    k = float(np.sum(np.abs(beta)) + 1e-12)
    w = beta / k
    alpha = np.asarray(alpha, float) * k                   # preserves alpha*beta'
    if w[np.argmax(np.abs(w))] < 0:                         # canonical sign (flip both)
        w = -w; alpha = -alpha
    return w, alpha


def basket_weights(Y, cfg):
    """Choose the spread-construction method. Always returns (w, alpha, method).

    - basket_method='vecm'    -> VECM; if it fails, fall back to Box-Tiao.
    - basket_method='boxtiao' -> Box-Tiao only.
    If every method fails, returns (None, None, None) and the basket is skipped.
    """
    if cfg.basket_method == 'vecm':
        w, alpha = vecm_weights(Y, cfg)
        if w is not None:
            return w, alpha, 'vecm'
        # --- fallback to Box-Tiao when the VECM fails ---
        w_bt = box_tiao_weights(Y)
        if w_bt is not None:
            return w_bt, None, 'boxtiao_fallback'
        return None, None, None

    if cfg.basket_method == 'boxtiao':
        w = box_tiao_weights(Y)
        if w is not None:
            return w, None, 'boxtiao'
        return None, None, None

    raise ValueError(f"unknown cfg.basket_method: {cfg.basket_method!r} "
                      f"(expected 'vecm' or 'boxtiao')")

## 5 · GARCH
Conditional volatility of the spread innovations: BIC selection over {GARCH, GJR, EGARCH} × {normal, t, skew-t} on formation data, then a causal forward variance with the parameters frozen. Same implementation as the pairs notebook.

In [ ]:
try:
    from arch import arch_model as _arch_model
    _HAS_ARCH = True
    try:
        from arch.utility.exceptions import ConvergenceWarning as _ArchConv
        warnings.simplefilter('ignore', _ArchConv)
    except Exception: pass
except Exception:
    _HAS_ARCH = False

_VOL_KW = {'GARCH': dict(vol='Garch', p=1, o=0, q=1),
           'GJR':   dict(vol='Garch', p=1, o=1, q=1),
           'EGARCH':dict(vol='EGARCH',p=1, o=1, q=1)}

def select_garch(vs_form, cfg):
    """BIC selection over volatility models x innovation distributions."""
    if not _HAS_ARCH: return None
    best = None
    for vol in cfg.garch_vols:
        for dist in cfg.garch_dists:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    res = _arch_model(vs_form, mean='Zero', dist=dist, **_VOL_KW[vol]).fit(disp='off')
                if best is None or res.bic < best['bic']:
                    best = dict(vol=vol, dist=dist, bic=float(res.bic), params=res.params)
            except Exception: continue
    return best

def _apply_garch_fixed(vs_full, vol, dist, params):
    am = _arch_model(vs_full, mean='Zero', dist=dist, **_VOL_KW[vol])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); fixed = am.fix(np.asarray(params))
    cv = np.asarray(fixed.conditional_volatility, float)
    med = np.nanmedian(cv[cv > 0]) if np.any(cv > 0) else 1.0
    cv = np.where(np.isfinite(cv) & (cv > 0), cv, med); return cv**2

def _garch_gauss(vs_form, vs_full):
    """Gaussian GARCH(1,1) fallback when `arch` is not available."""
    def nll(pr):
        o, al, be = pr
        if o <= 0 or al < 0 or be < 0 or al+be >= 0.999: return 1e12
        s2 = np.empty(len(vs_form)); s2[0] = vs_form.var()
        for t in range(1, len(vs_form)): s2[t] = o+al*vs_form[t-1]**2+be*s2[t-1]
        return 0.5*np.sum(np.log(2*np.pi*s2)+vs_form**2/s2)
    r = _minimize(nll, [vs_form.var()*0.05, 0.05, 0.90], method='Nelder-Mead')
    o, al, be = r.x; s2 = np.empty(len(vs_full)); s2[0] = o/max(1-al-be, 1e-6)
    for t in range(1, len(vs_full)): s2[t] = o+al*vs_full[t-1]**2+be*s2[t-1]
    return s2, dict(vol_model='GARCH', dist='normal')

def garch_sigma2(innov_full, f_end_idx, cfg):
    """Causal conditional variance of the innovations (model selected on formation)."""
    vs_full = innov_full*cfg.garch_scale
    vs_form = vs_full[cfg.kf_burnin:f_end_idx+1]
    sel = select_garch(vs_form, cfg)
    if sel is not None:
        s2 = _apply_garch_fixed(vs_full, sel['vol'], sel['dist'], sel['params'])/(cfg.garch_scale**2)
        return s2, dict(vol_model=sel['vol'], dist=sel['dist'])
    s2_scaled, meta = _garch_gauss(vs_form, vs_full)
    return s2_scaled/(cfg.garch_scale**2), meta

## 6 · HMM
Two-state Gaussian HMM on the **log realized volatility** of the spread (not on the z-score, to avoid circularity). Baum–Welch on formation data, causal Hamilton filter for $\phi_t$. Same implementation as the pairs notebook.

In [ ]:
def _gauss_B(y, mu, var, floor):
    var = np.maximum(var, floor)
    z = (y[:, None]-mu[None, :])**2/var[None, :]
    return np.exp(-0.5*z)/np.sqrt(2*np.pi*var[None, :])+1e-300

def _hmm_em(y, cfg, K=2, seed=0):
    """Baum-Welch (scaled forward-backward) for a K-state Gaussian HMM."""
    rng = np.random.default_rng(seed); fl = cfg.var_floor
    q = np.quantile(y, 0.5); lo, hi = y[y <= q], y[y > q]
    mu = np.array([lo.mean() if len(lo) else y.mean(), hi.mean() if len(hi) else y.mean()]) + rng.normal(0, 0.05, K)
    var = np.array([max(lo.var(), fl) if len(lo) else y.var(),
                    max(hi.var(), fl) if len(hi) else y.var()]) * rng.uniform(0.8, 1.2, K)
    A = np.array([[0.95, 0.05], [0.10, 0.90]]); pi = np.array([0.8, 0.2]); ll_old = -np.inf
    for _ in range(cfg.hmm_iters):
        B = _gauss_B(y, mu, var, fl); T = len(y)
        alpha = np.zeros((T, K)); c = np.zeros(T)
        alpha[0] = pi*B[0]; c[0] = alpha[0].sum()+1e-300; alpha[0] /= c[0]
        for t in range(1, T):
            alpha[t] = (alpha[t-1]@A)*B[t]; c[t] = alpha[t].sum()+1e-300; alpha[t] /= c[t]
        beta = np.zeros((T, K)); beta[-1] = 1.0
        for t in range(T-2, -1, -1): beta[t] = (A@(B[t+1]*beta[t+1]))/c[t+1]
        gamma = alpha*beta; gamma /= gamma.sum(1, keepdims=True)+1e-300
        xi = np.zeros((K, K))
        for t in range(T-1):
            m = (alpha[t][:, None]*A)*(B[t+1]*beta[t+1])[None, :]; xi += m/(m.sum()+1e-300)
        pi = gamma[0]; A = xi/(gamma[:-1].sum(0)[:, None]+1e-300); A /= A.sum(1, keepdims=True)
        g = gamma.sum(0)+1e-300; mu = (gamma*y[:, None]).sum(0)/g
        var = np.maximum((gamma*(y[:, None]-mu[None, :])**2).sum(0)/g, fl)
        ll = np.log(c).sum()
        if abs(ll-ll_old) < 1e-6: break
        ll_old = ll
    return dict(mu=mu, var=var, A=A, pi=pi, loglik=ll)

def _fit_hmm(y, cfg):
    best = None
    for s in range(cfg.hmm_restarts):
        try:
            m = _hmm_em(y, cfg, seed=s)
            if np.isfinite(m['loglik']) and (best is None or m['loglik'] > best['loglik']): best = m
        except Exception: continue
    return best

def _hamilton(y, mu, var, A, pi, floor):
    """Causal filtered state probabilities (Hamilton filter)."""
    B = _gauss_B(y, mu, var, floor); T, K = B.shape
    alpha = np.zeros((T, K)); a = pi*B[0]; alpha[0] = a/(a.sum()+1e-300)
    for t in range(1, T):
        a = (alpha[t-1]@A)*B[t]; alpha[t] = a/(a.sum()+1e-300)
    return alpha

## 7 · Signal helpers
Causal standardization of the z-score (rolling window, shifted by one day) and the entry / exit / stop / regime state machine, identical to the pairs notebook.

In [ ]:
def causal_standardize(x, win, minp):
    """Rolling z-score using only past data (expanding window during warm-up)."""
    s = pd.Series(np.asarray(x, float))
    mu = s.rolling(win, min_periods=minp).mean().shift(1)
    sd = s.rolling(win, min_periods=minp).std().shift(1)
    mu_e = s.expanding(min_periods=20).mean().shift(1)
    sd_e = s.expanding(min_periods=20).std().shift(1)
    mu = mu.where(sd.notna(), mu_e); sd = sd.where(sd.notna(), sd_e)
    return ((s-mu)/(sd+1e-12)).fillna(0.0).values

def gen_positions(z, phi, z_star, z_exit, z_stop, phi_star, hl, mh):
    """Position path in {-1, 0, +1} (long spread = +1). phi=None disables the regime gate."""
    n = len(z); pos = np.zeros(n, int); st = 0; en = 0
    cap = mh*hl if (hl and np.isfinite(hl)) else np.inf
    for t in range(n):
        zt = z[t]; pt = phi[t] if phi is not None else 1.0
        if st == 0:
            if zt < -z_star and pt > phi_star: st = +1; en = t
            elif zt > z_star and pt > phi_star: st = -1; en = t
        else:
            held = t-en
            if abs(zt) < z_exit or abs(zt) > z_stop or held > cap: st = 0
            elif (phi is not None) and pt < phi_star: st = 0
        pos[t] = st
    return pos

def _sharpe(r):
    r = np.asarray(r, float); s = r.std()
    return (r.mean()/s)*ANN if s > 1e-12 and len(r) > 5 else 0.0

## 8 · Stage T2 — Basket spreads + GARCH
For each candidate basket: take its legs, estimate the weights (VECM or Box–Tiao) on formation, apply them causally to the whole window and model the volatility of the spread changes with GARCH. Baskets are dropped if the half-life is outside [5, 150] days or the spread is degenerate. Outputs: spread series, weights $w$ and loadings $\alpha$.

Development runs use the windows whose trading period ends before `holdout_start`; holdout runs use the others (bounded by `holdout_end`, when set).

In [ ]:
def _is_holdout_window(win, cfg):
    """True if the trading end of window '<f0>_<t1>' falls in the holdout range."""
    t_end = pd.Timestamp(win.split('_')[-1])
    if t_end < pd.Timestamp(cfg.holdout_start):
        return False
    holdout_end = getattr(cfg, 'holdout_end', None)
    if holdout_end is not None and t_end > pd.Timestamp(holdout_end):
        return False
    return True


def _parse_legs(legs):
    """`legs` is a dict or the repr of a dict. Returns the list of tickers."""
    if isinstance(legs, dict): d = legs
    else:
        try: d = ast.literal_eval(str(legs))
        except Exception: return []
    return list(d.keys())

def model_baskets(cfg: Config, store: Store, force=False, holdout=False, verbose=True):
    """T2: weights, causal spread and GARCH variance per basket -> baskets/series, weights, meta."""
    if store.has('baskets/series') and not force:
        if verbose:
            bm = store.read('baskets/meta')
            print(f"[baskets] from cache: baskets={len(bm)} (force=False)")
        return store.read('baskets/series'), store.read('baskets/meta')
    if not _HAS_ARCH:
        print("[baskets] WARNING: `arch` not installed -> falling back to a Gaussian GARCH. Run `pip install arch`.")

    baskets = store.read_base('selection/baskets')
    # load prices once for the members of all baskets
    all_tk = set()
    for _, row in baskets.iterrows():
        all_tk.update(_parse_legs(row['legs']))
    close = store.load_close(tickers=all_tk); logclose = np.log(close)
    fdays = int(cfg.formation_years*252); emb = cfg.embargo_days

    series_rows, w_rows, meta_rows = [], [], []
    n = len(baskets); kept = 0
    for k, row in enumerate(baskets.itertuples(index=False), 1):
        win = row.window; group = row.group
        if holdout and not _is_holdout_window(win, cfg): continue
        if (not holdout) and _is_holdout_window(win, cfg): continue
        members = _parse_legs(row.legs)
        if len(members) < cfg.min_legs: continue
        members = [m for m in members if m in logclose.columns]
        if len(members) < cfg.min_legs: continue

        f0, t1 = pd.Timestamp(win.split('_')[0]), pd.Timestamp(win.split('_')[1])
        i0 = close.index.searchsorted(f0)
        if i0+fdays-1+emb >= len(close.index): continue
        f1 = close.index[i0+fdays-1]; e1 = close.index[i0+fdays-1+emb]
        df = logclose[members].loc[f0:t1].dropna()
        if len(df) < fdays//2: continue
        dts = df.index; Y = df.values
        f_mask = dts <= f1; f_end_idx = int(f_mask.sum()-1)
        if f_end_idx < cfg.kf_burnin+60: continue

        # --- spread construction on formation: VECM (Johansen) or Box-Tiao ---
        w, alpha, method = basket_weights(Y[f_mask], cfg)
        if w is None: continue
        # causal spread (frozen weights), centred on its formation mean
        s = Y @ w
        s = s - s[f_mask].mean()
        hl = half_life_of(s[f_mask])
        if not (cfg.half_life_min <= hl <= cfg.half_life_max): continue
        if np.std(s[f_mask]) < 1e-8: continue

        # innovation = spread change (input to GARCH and to the realized volatility)
        innov = np.r_[0.0, np.diff(s)]
        s2, gmeta = garch_sigma2(innov, f_end_idx, cfg)

        phase = np.where(dts <= f1, 'formation', np.where(dts <= e1, 'embargo', 'trading'))
        bid = f"BKT_{group.split('::')[-1].replace(' ', '')}_{win}"
        series_rows.append(pd.DataFrame({'pair_id': bid, 'date': dts, 'phase': phase,
                                         'spread': s, 'innov': innov, 'sigma2_S': s2}))
        for j, (tk, wi) in enumerate(zip(members, w)):
            al = float(alpha[j]) if (alpha is not None and j < len(alpha)) else np.nan
            w_rows.append(dict(pair_id=bid, ticker=tk, weight=float(wi), alpha_loading=al))
        # alpha_speed: norm of the loadings (aggregate speed of error correction)
        a_speed = float(np.sqrt(np.nansum(np.asarray(alpha, float)**2))) if alpha is not None else np.nan
        meta_rows.append(dict(pair_id=bid, window=win, group=group, n_legs=len(members),
                              half_life=hl, rank=getattr(row, 'rank', np.nan), method=method,
                              alpha_speed=a_speed,
                              vol_model=gmeta['vol_model'], dist=gmeta['dist'],
                              f_start=str(f0.date()), f_end=str(f1.date()),
                              t_start=str(e1.date()), t_end=str(t1.date()),
                              n_obs=len(dts), n_trading=int((phase == 'trading').sum())))
        kept += 1
        if verbose and (k % 10 == 0 or k == n):
            print(f"  [{k}/{n}] {bid} legs={len(members)} HL={hl:.1f} {method} {gmeta['vol_model']}-{gmeta['dist']}")

    series = pd.concat(series_rows, ignore_index=True) if series_rows else pd.DataFrame()
    weights = pd.DataFrame(w_rows); meta = pd.DataFrame(meta_rows)
    store.write('baskets/series', series, data_columns=['pair_id', 'phase'])
    store.write('baskets/weights', weights, data_columns=['pair_id', 'ticker'])
    store.write('baskets/meta', meta, data_columns=['pair_id', 'group'])
    if verbose:
        if len(meta):
            mc = meta['method'].value_counts().to_dict() if 'method' in meta.columns else {}
            print(f"[baskets] kept {kept}/{n} baskets | method={cfg.basket_method} {mc} | "
                  f"median HL={meta.half_life.median():.1f}d | mean legs={meta.n_legs.mean():.1f}")
        else:
            print(f"[baskets] kept 0/{n} baskets "
                  f"({'holdout: no basket with trading end >= '+cfg.holdout_start if holdout else 'no valid basket'}).")
    return series, meta

## 9 · Stage T3 — Regimes (HMM on each basket)
HMM on the realized volatility of the basket spread; the causal $\phi_t$ is used for gating.

In [ ]:
def detect_regimes_basket(cfg: Config, store: Store, force=False, verbose=True):
    """T3: fit the HMM per basket on formation data -> baskets/regimes."""
    if store.has('baskets/regimes') and not force:
        if verbose:
            print(f"[regimes] from cache (force=False)")
        return store.read('baskets/regimes')
    series = store.read('baskets/series')

    # --- guard: no basket survived model_baskets (e.g. empty holdout) ---
    if series is None or len(series) == 0 or 'pair_id' not in series.columns:
        if verbose: print("[regimes] no input baskets -- skipping.")
        empty = pd.DataFrame()
        store.write('baskets/regimes', empty, data_columns=['pair_id', 'phase'])
        return empty

    reg_rows = []; pair_ids = series.pair_id.unique()
    for k, pid in enumerate(pair_ids, 1):
        d = series[series.pair_id == pid].sort_values('date')
        sp = d['spread'].values; dts = d['date'].values; ph = d['phase'].values
        if len(sp) < 100: continue
        dv = np.r_[0.0, np.diff(sp)]
        rv = pd.Series(dv).rolling(cfg.rv_window, min_periods=cfg.rv_window).std().values
        valid = np.isfinite(rv) & (rv > 0)
        if valid.sum() < 80: continue
        y_full = np.full(len(rv), np.nan); y_full[valid] = np.log(rv[valid])
        form = (ph == 'formation') & valid
        if form.sum() < 60: continue
        y = (y_full-y_full[form].mean())/(y_full[form].std()+1e-12)
        first = np.where(valid)[0][0]; y[:first] = y[first]; y = np.nan_to_num(y, nan=0.0)
        m = _fit_hmm(y[ph == 'formation'], cfg)
        if m is None: continue
        stable = int(np.argmin(m['mu']))
        phi = _hamilton(y, m['mu'], m['var'], m['A'], m['pi'], cfg.var_floor)[:, stable]
        reg_rows.append(pd.DataFrame({'pair_id': pid, 'date': dts, 'phase': ph, 'phi': phi}))
        if verbose and (k % 10 == 0 or k == len(pair_ids)):
            tr = ph == 'trading'
            print(f"  [{k}/{len(pair_ids)}] {pid} phi_tr={phi[tr].mean() if tr.any() else float('nan'):.2f}")
    regimes = pd.concat(reg_rows, ignore_index=True) if reg_rows else pd.DataFrame()
    store.write('baskets/regimes', regimes, data_columns=['pair_id', 'phase'])
    if verbose:
        print(f"[regimes] baskets processed={regimes.pair_id.nunique() if len(regimes) else 0}")
    return regimes

## 10 · Stage T4 — Signals
Two causal z-scores per basket: `z` standardizes the spread by the GARCH conditional volatility (`static`, `full`), `z_raw` by the sample standard deviation only (`raw`). Both are stored with $\phi_t$.

In [ ]:
def make_signals_basket(cfg: Config, store: Store, force=False, verbose=True):
    """T4: trading-window z-scores (GARCH-scaled and raw) and phi -> baskets/signals."""
    if store.has('baskets/signals') and not force:
        if verbose: print(f"[signals] from cache (force=False)")
        return store.read('baskets/signals')
    series = store.read('baskets/series'); meta = store.read('baskets/meta')
    regimes = store.read('baskets/regimes')

    if (series is None or len(series) == 0 or 'pair_id' not in series.columns
            or regimes is None or len(regimes) == 0 or 'pair_id' not in regimes.columns):
        if verbose: print("[signals] no upstream input -- skipping.")
        empty = pd.DataFrame()
        store.write('baskets/signals', empty, data_columns=['pair_id'])
        return empty

    phi_map = regimes.set_index(['pair_id', 'date'])['phi']
    out_rows = []
    pids = series.pair_id.unique()
    for k, pid in enumerate(pids, 1):
        d = series[series.pair_id == pid].sort_values('date').reset_index(drop=True)
        dts = d['date'].values; ph = d['phase'].values
        s = d['spread'].values; s2 = d['sigma2_S'].values; f = d['innov'].values
        if (ph == 'formation').sum() < 60: continue
        # z (GARCH-adaptive): spread scaled by the GARCH conditional volatility
        z_garch = causal_standardize(s/np.sqrt(s2*cfg.rv_window+1e-12), cfg.std_win, cfg.std_minp)
        # z_raw: spread scaled by the sample standard deviation only (no adaptive
        # denominator). Baseline analogous to 'ols' in the pairs branch; it lets
        # the GARCH effect be separated from the HMM effect.
        z_raw = causal_standardize(s, cfg.std_win, cfg.std_minp)
        phi = np.array([phi_map.get((pid, dt), np.nan) for dt in dts])
        phi = pd.Series(phi).ffill().fillna(1.0).values
        tr = (ph == 'trading'); idx = np.where(tr)[0]
        if len(idx) < 5: continue
        sl = slice(idx[0], idx[-1]+1)
        out_rows.append(pd.DataFrame({'pair_id': pid, 'date': dts[sl],
                                      'z': z_garch[sl], 'z_raw': z_raw[sl], 'phi': phi[sl]}))
        if verbose and (k % 10 == 0 or k == len(pids)):
            print(f"  [{k}/{len(pids)}] {pid}")
    signals = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()
    store.write('baskets/signals', signals, data_columns=['pair_id'])
    if verbose:
        if len(signals):
            print(f"[signals] baskets={signals.pair_id.nunique()} rows={len(signals):,} | "
                  f"std z_garch={signals.z.std():.2f} std z_raw={signals.z_raw.std():.2f}")
        else:
            print("[signals] no basket survived.")
    return signals

## 11 · N-leg P&L and thresholds
Per-basket panel (z-scores, $\phi$, weights, leg log-prices). P&L: $r_t=\tau_{t-1}\sum_i w_i \Delta y_{i,t}$, cost $c\,|\Delta\tau_t|\sum_i|w_i|$.

`optimize_global_basket` can calibrate the thresholds on the baskets themselves, but with a few dozen baskets this overfits badly (high PBO). The default, `_freeze_pair_thresholds` (Section 13), reuses the thresholds of the pairs branch instead.

In [ ]:
def _build_basket_panel(cfg, store):
    """Per-basket arrays (z, z_raw, phi, weights, leg log-prices) for the backtests."""
    sig = store.read('baskets/signals'); meta = store.read('baskets/meta')
    weights = store.read('baskets/weights')

    if (sig is None or len(sig) == 0 or 'pair_id' not in sig.columns
            or weights is None or len(weights) == 0):
        return {}, np.array([], dtype='datetime64[ns]'), {}

    hl_map = meta.set_index('pair_id')['half_life'].to_dict()
    all_tk = weights.ticker.unique()
    close = store.load_close(tickers=all_tk); lc = np.log(close)
    P = {}
    for pid in sig.pair_id.unique():
        g = sig[sig.pair_id == pid].sort_values('date')
        wsub = weights[weights.pair_id == pid]
        members = wsub.ticker.tolist(); w = wsub.weight.values
        Yp = pd.DataFrame({'date': g.date.values}).merge(
            pd.DataFrame({'date': lc.index, **{m: lc[m].values for m in members}}),
            on='date', how='left')
        Ymat = Yp[members].values
        # backward compatibility: caches written before z_raw existed fall back to z
        z_raw_vals = g['z_raw'].values if 'z_raw' in g.columns else g.z.values
        P[pid] = dict(date=g.date.values, z=g.z.values, z_raw=z_raw_vals, phi=g.phi.values,
                      w=w, Y=Ymat, hl=hl_map.get(pid, np.nan), wabs=float(np.sum(np.abs(w))))
    all_dates = (np.array(sorted(set(np.concatenate([p['date'] for p in P.values()]))))
                 if P else np.array([], dtype='datetime64[ns]'))
    return P, all_dates, {d: i for i, d in enumerate(all_dates)}


def _basket_net_returns(p, spec, zs, ze, cfg, c):
    """Net daily returns of one basket. spec in {'raw', 'static', 'full'}.
      - 'raw'    : fixed weights, z scaled by the sample std (no GARCH, no HMM)
      - 'static' : fixed weights, z scaled by the GARCH conditional volatility (no HMM)
      - 'full'   : 'static' + HMM gate on phi_t
    Returns (net, pos)."""
    z = p['z_raw'] if spec == 'raw' else p['z']
    phi = p['phi'] if spec == 'full' else None
    ps = cfg.phi_star if spec == 'full' else -1.0
    pos = gen_positions(z, phi, zs, ze, cfg.z_stop, ps, p['hl'], cfg.max_hold_mult)
    Y = p['Y']; w = p['w']                              # (T,N), (N,)
    dY = np.zeros_like(Y); dY[1:] = Y[1:]-Y[:-1]        # log-price changes per leg
    spread_ret = dY @ w                                 # sum_i w_i dy_i,t
    gross = np.r_[0.0, pos[:-1]*spread_ret[1:]]
    turn = np.r_[0.0, np.abs(np.diff(pos))*p['wabs']]   # cost proportional to gross notional
    return gross - c*turn, pos

def _metrics_b(net, pos):
    """Performance metrics of a daily net-return series."""
    net = np.asarray(net); eq = np.cumsum(net); dd = eq-np.maximum.accumulate(eq)
    sd = net.std()+1e-12; ntr = int((np.abs(np.diff(pos)) > 0).sum())//2; maxdd = dd.min()
    return dict(sharpe=net.mean()/sd*ANN, ann_ret=net.mean()*252, total=eq[-1], maxDD=maxdd,
                calmar=(net.mean()*252)/abs(maxdd) if maxdd < -1e-9 else np.nan,
                cvar95=net[net <= np.percentile(net, 5)].mean() if (net < 0).any() else 0.0,
                n_trades=ntr, exposure=(pos != 0).mean(),
                hit=(net[net != 0] > 0).mean() if (net != 0).any() else np.nan)


def _cscv_pbo(R, S):
    """Probability of backtest overfitting via CSCV."""
    N, T = R.shape
    if T < S*5: return np.nan
    bsz = T//S; blocks = [np.arange(i*bsz, (i+1)*bsz if i < S-1 else T) for i in range(S)]
    lams = []
    for IS in itertools.combinations(range(S), S//2):
        OOS = [b for b in range(S) if b not in IS]
        ii = np.concatenate([blocks[b] for b in IS]); oo = np.concatenate([blocks[b] for b in OOS])
        sr_is = np.array([_sharpe(R[n, ii]) for n in range(N)])
        sr_oos = np.array([_sharpe(R[n, oo]) for n in range(N)])
        ns = int(np.argmax(sr_is)); rank = (sr_oos.argsort().argsort()[ns]+1)/(N+1)
        rank = min(max(rank, 1e-6), 1-1e-6); lams.append(np.log(rank/(1-rank)))
    return float((np.array(lams) <= 0).mean())

def _portfolio_net_b(P, all_dates, dpos, spec, zs, ze, cfg, c):
    S = np.zeros(len(all_dates)); Cnt = np.zeros(len(all_dates))
    for pid, p in P.items():
        net, pos = _basket_net_returns(p, spec, zs, ze, cfg, c)
        ii = np.array([dpos[d] for d in p['date']]); S[ii] += net; Cnt[ii] += (np.r_[0, pos[:-1]] != 0)
    return S/np.maximum(Cnt, 1)

def optimize_global_basket(cfg: Config, store: Store, force=False, verbose=True):
    """Optional: grid search of (z*, z_exit) on the baskets themselves (overfitting risk)."""
    if store.has('baskets/thresholds') and not force:
        thr = store.read('baskets/thresholds')
        if verbose: print(f"[optimize] from cache:\n{thr.to_string(index=False)}")
        return thr
    P, all_dates, dpos = _build_basket_panel(cfg, store); c = cfg.cost_bps/1e4

    # --- guard: no basket available, thresholds cannot be computed ---
    if not P:
        if verbose: print("[optimize] no basket available -- skipping (thresholds not computable).")
        thr = pd.DataFrame(columns=['spec', 'z_star', 'z_exit', 'phi_star', 'z_stop',
                                     'max_hold_mult', 'cost_bps', 'sharpe_net_dev', 'pbo', 'n_trials'])
        store.write('baskets/thresholds', thr)
        return thr

    results = {}
    for spec in ['raw', 'static', 'full']:
        cands = [(zs, ze) for zs in cfg.grid_zstar for ze in cfg.grid_zexit]
        R = np.array([_portfolio_net_b(P, all_dates, dpos, spec, zs, ze, cfg, c) for (zs, ze) in cands])
        srs = np.array([_sharpe(R[i]) for i in range(len(cands))])
        best = int(np.argmax(srs)); zs, ze = cands[best]
        pbo = _cscv_pbo(R, cfg.cscv_blocks)
        results[spec] = dict(z_star=zs, z_exit=ze,
                             phi_star=(cfg.phi_star if spec == 'full' else np.nan),
                             z_stop=cfg.z_stop, max_hold_mult=cfg.max_hold_mult, cost_bps=cfg.cost_bps,
                             sharpe_net_dev=float(srs[best]), pbo=pbo, n_trials=len(cands))
        if verbose:
            print(f"[{spec.upper():6s}] z*={zs} z_exit={ze} "
                  f"phi*={cfg.phi_star if spec=='full' else '—'} | SR_net={srs[best]:.2f} PBO={pbo:.2f}")
    thr = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'spec'})
    store.write('baskets/thresholds', thr)
    return thr

## 12 · Quality ranking and backtest
Baskets are ranked by the sum of two cross-sectional z-scores: distance of the half-life from the 25-day target and the mean regime probability $\phi$. The backtest compares raw / static / full at the frozen thresholds, per basket and for the fixed-capital (1/N) portfolio; `backtest_invested_baskets` gives the per-window invested view, as in the pairs notebook.

> **Caveat.** As in the pairs notebook, the regime component is the average $\phi$ over the *trading* window, so the ranking (not the positions) uses information that is not available at the start of the window.

In [ ]:
def quality_basket(cfg: Config, store: Store, force=False, verbose=True):
    """Rank baskets by half-life distance from target and mean phi -> baskets/quality."""
    if store.has('baskets/quality') and not force:
        q = store.read('baskets/quality')
        if verbose: print(f"[quality] from cache: {len(q)} baskets")
        return q
    meta = store.read('baskets/meta')

    if meta is None or len(meta) == 0:
        if verbose: print("[quality] no input baskets -- skipping.")
        empty = pd.DataFrame()
        store.write('baskets/quality', empty, data_columns=['pair_id'])
        return empty

    meta = meta.copy()
    reg = store.read('baskets/regimes')
    if reg is not None and len(reg) and 'pair_id' in reg.columns:
        phi_tr = (reg[reg.phase == 'trading'].groupby('pair_id')['phi'].mean()
                  .rename('phi_mean_trading'))
        meta = meta.merge(phi_tr, on='pair_id', how='left')
    else:
        meta['phi_mean_trading'] = np.nan
    def z(s): s = pd.to_numeric(s, errors='coerce'); return (s-s.mean())/(s.std()+1e-12)
    score = (z(-(np.log(meta['half_life']/cfg.half_life_target)).abs())
             + z(meta['phi_mean_trading'].fillna(meta['phi_mean_trading'].median())))
    meta['quality_score'] = score
    meta = meta.sort_values('quality_score', ascending=False).reset_index(drop=True)
    meta['rank'] = np.arange(1, len(meta)+1)
    store.write('baskets/quality', meta, data_columns=['pair_id'])
    if verbose:
        print(f"[quality] {len(meta)} baskets. TOP {min(cfg.top_n,len(meta))}:")
        print(meta[['rank', 'pair_id', 'quality_score', 'half_life', 'phi_mean_trading']]
              .head(cfg.top_n).to_string(index=False))
    return meta


# ═════════════════════════════════════════════════════════════════════════════
# BASKET BACKTEST — raw vs static vs full, per basket + fixed-capital (1/N) portfolio
# ═════════════════════════════════════════════════════════════════════════════
_BASKET_SPECS = ['raw', 'static', 'full']
_BASKET_LABELS = {'raw': 'Box-Tiao/VECM (raw)', 'static': '+ GARCH', 'full': '+ GARCH + HMM gating'}


def backtest_baskets(cfg: Config, store: Store, top_n=None, per_basket=True, plot=True, verbose=True):
    """Flat view: per-basket results for the top-N baskets and the fixed-capital portfolio."""
    top_n = top_n or cfg.top_n
    thr = store.read('baskets/thresholds').set_index('spec')
    qual = store.read('baskets/quality')

    if qual is None or len(qual) == 0:
        if verbose: print("[backtest] no baskets available (empty quality ranking).")
        return {}, pd.DataFrame()

    P, all_dates, dpos = _build_basket_panel(cfg, store); c = cfg.cost_bps/1e4
    if not P:
        if verbose: print("[backtest] no baskets available (empty panel).")
        return {}, pd.DataFrame()

    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    if verbose and len(top_ids) < len(ranked):
        print(f"[backtest] {len(ranked)-len(top_ids)} baskets dropped (insufficient data); using {len(top_ids)}.")
    if not top_ids:
        print("[backtest] no baskets available."); return {}, pd.DataFrame()

    per_basket_res = {}
    if per_basket:
        try:
            import matplotlib.pyplot as plt; have_plt = True
        except Exception:
            have_plt = False; plot = False
        for pid in top_ids:
            p = P[pid]; rows = {}; curves = {}
            for spec in _BASKET_SPECS:
                zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
                net, pos = _basket_net_returns(p, spec, zs, ze, cfg, c)
                rows[spec] = _metrics_b(net, pos)
                curves[spec] = (pd.to_datetime(p['date']), np.cumsum(net))
            res = pd.DataFrame(rows).T; per_basket_res[pid] = res
            if verbose:
                print(f"\n{'='*64}\n {pid}  (legs={len(p['w'])})\n{'='*64}")
                print(res[['sharpe', 'ann_ret', 'maxDD', 'calmar', 'cvar95', 'n_trades', 'exposure', 'hit']].round(3).to_string())
            if plot and have_plt:
                _C = globals().get('C', {})
                col = {'raw': _C.get('normal', '#444444'), 'static': _C.get('bull', '#1f77b4'), 'full': _C.get('gfc', '#d62728')}
                fig, ax = plt.subplots(figsize=(10, 3.2))
                for s in _BASKET_SPECS:
                    dts, eq = curves[s]; ax.plot(dts, eq, color=col[s], lw=1.6, label=_BASKET_LABELS[s])
                ax.set_title(f'{pid} — net cumulative P&L', fontweight='bold')
                ax.set_ylabel('cum. net P&L'); ax.legend(frameon=False); ax.axhline(0, color='k', lw=.6)
                plt.tight_layout(); plt.show()

    # fixed-capital (1/N) portfolio
    N = len(top_ids); agg = {}
    for spec in _BASKET_SPECS:
        zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
        S = np.zeros(len(all_dates)); POS = np.zeros(len(all_dates))
        for pid in top_ids:
            net, pos = _basket_net_returns(P[pid], spec, zs, ze, cfg, c)
            ii = np.array([dpos[d] for d in P[pid]['date']]); S[ii] += net; POS[ii] += (pos != 0)
        r = S/max(N, 1); eq = np.cumsum(r); dd = eq-np.maximum.accumulate(eq)
        agg[spec] = dict(sharpe=_sharpe(r), ann_ret=r.mean()*252, maxDD=dd.min(),
                         calmar=(r.mean()*252)/abs(dd.min()) if dd.min() < -1e-9 else np.nan,
                         cvar95=r[r <= np.percentile(r, 5)].mean() if (r < 0).any() else 0.0,
                         exposure=(POS/max(N, 1)).mean())
    aggdf = pd.DataFrame(agg).T
    if verbose:
        print(f"\n{'='*64}\n TOP-{top_n} BASKET PORTFOLIO (fixed capital 1/N)\n{'='*64}")
        print(aggdf[['sharpe', 'ann_ret', 'maxDD', 'calmar', 'cvar95', 'exposure']].round(3).to_string())
    return per_basket_res, aggdf


def backtest_invested_baskets(cfg: Config, store: Store, top_n=None, verbose=True, plot=False):
    """Invested view: per-window allocation, chained equity (as in the pairs notebook)."""
    top_n = top_n or cfg.top_n
    thr = store.read('baskets/thresholds').set_index('spec')
    qual = store.read('baskets/quality')

    if qual is None or len(qual) == 0:
        if verbose: print("[backtest_invested] no baskets available (empty quality ranking).")
        return pd.DataFrame(), {}

    P, _, _ = _build_basket_panel(cfg, store); c = cfg.cost_bps/1e4
    if not P:
        if verbose: print("[backtest_invested] no baskets available (empty panel).")
        return pd.DataFrame(), {}

    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    if not top_ids:
        if verbose: print("[backtest_invested] no baskets available.")
        return pd.DataFrame(), {}

    from collections import defaultdict
    by_win = defaultdict(list)
    for pid in top_ids:
        by_win['_'.join(pid.split('_')[-2:])].append(pid)
    wins = sorted(by_win, key=lambda w: w.split('_')[-1])
    out = {}; curves = {}
    for spec in _BASKET_SPECS:
        zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
        seg_r, seg_d = [], []
        for win in wins:
            pids = by_win[win]
            wd = np.array(sorted(set(np.concatenate([P[pid]['date'] for pid in pids]))))
            wpos = {d: i for i, d in enumerate(wd)}
            S = np.zeros(len(wd)); ACT = np.zeros(len(wd))
            for pid in pids:
                net, pos = _basket_net_returns(P[pid], spec, zs, ze, cfg, c)
                ii = np.array([wpos[d] for d in P[pid]['date']])
                S[ii] += net; ACT[ii] += (np.r_[0, pos[:-1]] != 0)
            seg_r.append(np.where(ACT > 0, S/np.maximum(ACT, 1), 0.0)); seg_d.append(wd)
        r = np.concatenate(seg_r); dts = pd.to_datetime(np.concatenate(seg_d))
        order = np.argsort(dts.values); r = r[order]; dts = dts[order]
        eq = np.cumsum(r); dd = eq-np.maximum.accumulate(eq)
        out[spec] = dict(sharpe=_sharpe(r), ann_ret=r.mean()*252, total=eq[-1], maxDD=dd.min(),
                         calmar=(r.mean()*252)/abs(dd.min()) if dd.min() < -1e-9 else np.nan,
                         frac_active_days=float((r != 0).mean()))
        curves[spec] = (dts, eq)
    res = pd.DataFrame(out).T
    if verbose:
        print(f"\n{'='*68}\n INVESTED BASKET BACKTEST — top-{top_n}, per-window allocation\n{'='*68}")
        print(res[['sharpe', 'ann_ret', 'total', 'maxDD', 'calmar', 'frac_active_days']].round(3).to_string())
    if plot:
        try:
            import matplotlib.pyplot as plt
            _C = globals().get('C', {})
            col = {'raw': _C.get('normal', '#444444'), 'static': _C.get('bull', '#1f77b4'), 'full': _C.get('gfc', '#d62728')}
            fig, ax = plt.subplots(figsize=(11, 4))
            for s in _BASKET_SPECS:
                dts, eq = curves[s]; ax.plot(dts, eq, color=col[s], lw=1.6, label=_BASKET_LABELS[s])
            ax.set_title(f'Baskets — chained equity (invested view, top-{top_n})', fontweight='bold')
            ax.set_ylabel('cum. net P&L'); ax.legend(frameon=False); ax.axhline(0, color='k', lw=.6)
            plt.tight_layout(); plt.show()
        except Exception: pass
    return res, curves

## 13 · Orchestrators
* `_freeze_pair_thresholds` — copies the pairs thresholds with a scale-consistent mapping: `raw ← ols` (static sample z-score), `static`/`full` ← `kf` (adaptive denominator). The HMM gate is then the only difference between `static` and `full`.
* `run_baskets` — development chain.
* `run_baskets_holdout` — holdout chain with frozen thresholds in the `holdout` namespace.

### 13b · Second independent holdout: the 2008–2009 crisis
The main holdout (trading windows ending from 2020) contains no cointegrated basket, so it gives no out-of-sample evidence for this branch. The functions below repeat the protocol of Section 12.9 of the pairs notebook: an independent development sample ending on 2007-06-30, frozen thresholds, a single test on 2008–2009. The thresholds come from the pairs notebook (namespace `crisis2008_dev`); if they are missing, they are optimized on the pre-crisis baskets, with the overfitting risk noted above.

In [ ]:
def _freeze_pair_thresholds(cfg, store, verbose=True):
    """Reuse the pairs thresholds: raw <- ols (static sample z-score),
    static/full <- kf (adaptive denominator). Writes baskets/thresholds."""
    if not store.has('params/thresholds'):
        raise RuntimeError("params/thresholds (pairs) missing: run the pairs pipeline first, "
                           "or use optimize=True to optimize on the baskets.")
    pt = store.read('params/thresholds').set_index('spec')
    zs_ols, ze_ols = float(pt.loc['ols', 'z_star']), float(pt.loc['ols', 'z_exit'])
    zs_kf, ze_kf = float(pt.loc['kf', 'z_star']), float(pt.loc['kf', 'z_exit'])
    thr = pd.DataFrame([
        dict(spec='raw', z_star=zs_ols, z_exit=ze_ols, phi_star=np.nan, z_stop=cfg.z_stop,
             max_hold_mult=cfg.max_hold_mult, cost_bps=cfg.cost_bps, sharpe_net_dev=np.nan,
             pbo=np.nan, n_trials=0),
        dict(spec='static', z_star=zs_kf, z_exit=ze_kf, phi_star=np.nan, z_stop=cfg.z_stop,
             max_hold_mult=cfg.max_hold_mult, cost_bps=cfg.cost_bps, sharpe_net_dev=np.nan,
             pbo=np.nan, n_trials=0),
        dict(spec='full', z_star=zs_kf, z_exit=ze_kf, phi_star=cfg.phi_star, z_stop=cfg.z_stop,
             max_hold_mult=cfg.max_hold_mult, cost_bps=cfg.cost_bps, sharpe_net_dev=np.nan,
             pbo=np.nan, n_trials=0),
    ])
    store.write('baskets/thresholds', thr)
    if verbose:
        print(f"[thresholds] reusing pairs thresholds: raw<-ols(z*={zs_ols},z_exit={ze_ols}) "
              f"static/full<-kf(z*={zs_kf},z_exit={ze_kf}, full phi*={cfg.phi_star})")
    return thr


def run_baskets(cfg: Config, force_from=None, top_n=None, optimize=False):
    """Development run. optimize=False (default) reuses the frozen pairs thresholds;
    optimize=True calibrates them on the baskets (overfitting risk)."""
    order = ['baskets', 'regimes', 'signals', 'optimize', 'quality']
    fi = order.index(force_from) if force_from in order else len(order)
    store = Store(cfg)
    model_baskets(cfg, store, force=(fi <= 0), holdout=False)
    detect_regimes_basket(cfg, store, force=(fi <= 1))
    make_signals_basket(cfg, store, force=(fi <= 2))
    if optimize:
        optimize_global_basket(cfg, store, force=(fi <= 3))
    else:
        _freeze_pair_thresholds(cfg, store)
    quality_basket(cfg, store, force=(fi <= 4))
    return backtest_baskets(cfg, store, top_n=top_n)

def run_baskets_holdout(cfg: Config, force=False, top_n=None, verbose=True):
    """Holdout run with the development thresholds frozen."""
    dev = Store(cfg); hold = Store(cfg, ns='holdout')
    # frozen thresholds: copy the development ones
    if dev.has('baskets/thresholds'):
        hold.write('baskets/thresholds', dev.read('baskets/thresholds'))
    elif dev.has('params/thresholds'):
        _freeze_pair_thresholds(cfg, hold, verbose=verbose)
    else:
        raise RuntimeError("thresholds missing: run run_baskets (development) first.")
    if verbose:
        print("=== BASKET HOLDOUT | thresholds FROZEN ===")
    model_baskets(cfg, hold, force=force, holdout=True, verbose=verbose)
    detect_regimes_basket(cfg, hold, force=force, verbose=verbose)
    make_signals_basket(cfg, hold, force=force, verbose=verbose)
    quality_basket(cfg, hold, force=force, verbose=verbose)
    return backtest_baskets(cfg, hold, top_n=top_n, verbose=verbose)

In [ ]:
from dataclasses import replace


def run_baskets_ns(cfg: Config, ns=None, force_from=None, top_n=None, optimize=False):
    """run_baskets() with an explicit namespace, so the main results (ns=None) are never overwritten."""
    order = ['baskets', 'regimes', 'signals', 'optimize', 'quality']
    fi = order.index(force_from) if force_from in order else len(order)
    store = Store(cfg, ns=ns)
    model_baskets(cfg, store, force=(fi <= 0), holdout=False)
    detect_regimes_basket(cfg, store, force=(fi <= 1))
    make_signals_basket(cfg, store, force=(fi <= 2))
    if optimize:
        optimize_global_basket(cfg, store, force=(fi <= 3))
    else:
        _freeze_pair_thresholds(cfg, store)
    quality_basket(cfg, store, force=(fi <= 4))
    return backtest_baskets(cfg, store, top_n=top_n)


def run_baskets_holdout_ns(cfg: Config, dev_ns=None, hold_ns='holdout', force=False, top_n=None, verbose=True):
    """run_baskets_holdout() with explicit namespaces for the development (threshold source) and the holdout."""
    dev = Store(cfg, ns=dev_ns); hold = Store(cfg, ns=hold_ns)
    if dev.has('baskets/thresholds'):
        hold.write('baskets/thresholds', dev.read('baskets/thresholds'))
    elif dev.has('params/thresholds'):
        _freeze_pair_thresholds(cfg, hold, verbose=verbose)
    else:
        raise RuntimeError(f"thresholds missing in namespace '{dev_ns}': "
                           f"run run_baskets_ns(cfg, ns='{dev_ns}') first.")
    if verbose:
        print(f"=== BASKET HOLDOUT (ns='{hold_ns}') | thresholds FROZEN from ns='{dev_ns}' ===")
    model_baskets(cfg, hold, force=force, holdout=True, verbose=verbose)
    detect_regimes_basket(cfg, hold, force=force, verbose=verbose)
    make_signals_basket(cfg, hold, force=force, verbose=verbose)
    quality_basket(cfg, hold, force=force, verbose=verbose)
    return backtest_baskets(cfg, hold, top_n=top_n, verbose=verbose)


def run_basket_crisis_holdout_2008(base_cfg: Config, pre_crisis_dev_end='2007-06-30',
                                    crisis_start='2008-01-01', crisis_end='2009-12-31',
                                    top_n=None, reuse_pairs_thresholds=True, verbose=True):
    """2008-2009 experiment for the basket branch: pre-crisis development
    (up to `pre_crisis_dev_end`) -> frozen thresholds -> single test on the crisis.

    With reuse_pairs_thresholds=True the thresholds come from the pairs notebook
    (namespace 'crisis2008_dev' of the same HDF5 file). If they are missing, or
    reuse_pairs_thresholds=False, they are optimized on the pre-crisis baskets.
    """
    top_n = top_n or base_cfg.top_n
    cfg_crisis = replace(base_cfg, dev_end=pre_crisis_dev_end,
                         holdout_start=crisis_start, holdout_end=crisis_end)

    dev_ns = 'crisis2008_dev'
    pairs_thr_available = Store(cfg_crisis, ns=dev_ns).has('params/thresholds')
    use_optimize = not (reuse_pairs_thresholds and pairs_thr_available)

    if verbose:
        print("=== 2008-2009 CRISIS EXPERIMENT (baskets) ===")
        print(f"Development (thresholds): data up to {pre_crisis_dev_end}")
        print(f"Holdout (frozen thresholds, single test): {crisis_start} -> {crisis_end}")
        if use_optimize:
            print("[!] Pairs thresholds (ns='crisis2008_dev') not found or reuse_pairs_thresholds=False "
                  "-> optimizing on the pre-crisis baskets (overfitting risk with few baskets).")
        else:
            print("Thresholds: inherited from the pairs branch (ns='crisis2008_dev').")

    _, agg_pre_crisis_dev = run_baskets_ns(cfg_crisis, ns=dev_ns, top_n=top_n, optimize=use_optimize)
    if verbose:
        print(f"\n=== PRE-CRISIS DEVELOPMENT, BASKETS (up to {pre_crisis_dev_end}) ===")
        print(agg_pre_crisis_dev.round(3).to_string() if not agg_pre_crisis_dev.empty else "(empty)")

    _, agg_crisis_hold = run_baskets_holdout_ns(cfg_crisis, dev_ns=dev_ns,
                                                hold_ns='crisis2008_holdout', force=True, top_n=top_n)
    if verbose:
        print(f"\n=== CRISIS HOLDOUT, BASKETS ({crisis_start} -> {crisis_end}) ===")
        print(agg_crisis_hold.round(3).to_string() if not agg_crisis_hold.empty
              else "(empty -- no cointegrated basket survives in 2008-2009 either)")

    return agg_pre_crisis_dev, agg_crisis_hold


def basket_crisis_summary(agg_pre_crisis_dev, agg_crisis_hold):
    """Two-column net Sharpe table: pre-crisis development vs crisis holdout."""
    specs = ['raw', 'static', 'full']

    def _col(df, label):
        if df is None or len(df) == 0 or 'sharpe' not in df.columns:
            print(f"[basket_crisis_summary] '{label}' is empty -- column filled with NaN.")
            return pd.Series(np.nan, index=specs)
        return df.reindex(specs)['sharpe']

    out = pd.DataFrame({
        'pre_crisis_dev (up to 2007)': _col(agg_pre_crisis_dev, 'agg_pre_crisis_dev'),
        'crisis_holdout (2008-2009)': _col(agg_crisis_hold, 'agg_crisis_hold'),
    })
    print(f"\n{'='*64}\n BASKETS: 2008-2009 CRISIS EXPERIMENT (only out-of-sample evidence)\n{'='*64}")
    print(out.round(3).to_string())
    return out

## 14 · Experiments
### 14.1 · Development
Set `hdf_file`. With `basket_method='vecm'` the spread is the first Johansen cointegrating vector. `sync_to_drive` saves the local working file back to Drive at the end of each stage.

In [ ]:
cfg = Config(hdf_file='/content/drive/MyDrive/data_ip_2026_v2.h5', basket_method='vecm',
             top_n=10, work_local=True)
per_basket, agg = run_baskets(cfg, force_from='baskets', top_n=10)
print('\n=== BASKET DEV (VECM) ==='); print(agg.round(3).to_string())
sync_to_drive(cfg)

### 14.2 · Holdout (trading windows ending from 2020-01-01)
With the thesis data no basket passes screening in this period; the chain then ends cleanly with an empty result instead of an error.

In [ ]:
per_basket_h, agg_h = run_baskets_holdout(cfg, force=True, top_n=10)
print('\n=== BASKET HOLDOUT (VECM) ===')
if agg_h.empty:
    print('No basket available in the holdout: no cointegrated basket survives screening '
          'in the out-of-sample period with this universe and these thresholds.')
else:
    print(agg_h.round(3).to_string())
sync_to_drive(cfg)

### 14.3 · raw vs static vs full (development)
`raw → static` isolates the GARCH denominator, `static → full` the HMM gate.

In [ ]:
specs = ['raw', 'static', 'full']
labels = ['Box-Tiao/VECM (raw)', '+ GARCH', '+ GARCH + HMM gating']
bar_c = [C['normal'], C['bull'], C['gfc']]
fig, axes = plt.subplots(1, 2, figsize=(11, 4)); x = np.arange(3); w = 0.5

axes[0].bar(x, agg.loc[specs, 'sharpe'].values, w, color=bar_c, alpha=0.9, edgecolor='k', linewidth=0.6)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=8); axes[0].axhline(0, color='k', lw=.7)
axes[0].set_ylabel('Sharpe (annualized)'); axes[0].set_title('Net Sharpe: raw vs static vs full')

axes[1].bar(x, 100*agg.loc[specs, 'maxDD'].values, w, color=bar_c, alpha=0.9, edgecolor='k', linewidth=0.6)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=8); axes[1].axhline(0, color='k', lw=.7)
axes[1].set_ylabel('max drawdown (%)'); axes[1].set_title('Max drawdown: raw vs static vs full')

fig.suptitle('Basket VECM–GARCH–HMM: incremental effect of GARCH and HMM', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR + 'basket_static_vs_full.png'); plt.show()

### 14.4 · VECM loadings $\alpha$
The loadings show which leg of each basket drives the correction towards equilibrium. A large $|\alpha|$ means the asset moves a lot to close the gap; a value near zero means the asset is weakly exogenous (it does not adjust).

In [ ]:
store = Store(cfg)
W = store.read('baskets/weights')
M = store.read('baskets/meta')
if 'alpha_loading' in W.columns:
    top = M.nsmallest(5, 'rank')['pair_id'].tolist() if 'rank' in M.columns else M['pair_id'].head(5).tolist()
    for pid in top:
        sub = W[W.pair_id == pid]
        print(f'\n{pid}')
        print(sub[['ticker', 'weight', 'alpha_loading']].round(3).to_string(index=False))
else:
    print('Alpha loadings not available (Box-Tiao method). Use basket_method=\'vecm\'.')

### 14.5 · Second holdout: the 2008–2009 crisis
The only out-of-sample evidence for this branch. Requires Section 12.9 of the pairs notebook to have been run on the same HDF5 file.

In [ ]:
agg_pre_crisis_dev, agg_crisis_hold = run_basket_crisis_holdout_2008(
    cfg,
    pre_crisis_dev_end='2007-06-30',
    crisis_start='2008-01-01',
    crisis_end='2009-12-31',
    top_n=10,
    reuse_pairs_thresholds=True,
)

In [ ]:
basket_crisis_table = basket_crisis_summary(agg_pre_crisis_dev, agg_crisis_hold)
sync_to_drive(cfg)

## 15 · Summary
Results reported in Section 13 of the thesis (thresholds inherited from the pairs branch, 5 bps per side):

* **Development.** 21 of 33 candidate baskets are admitted. On the Top-10 portfolio the net Sharpe rises monotonically from `raw` to `full` and `full` roughly halves the maximum drawdown of `raw`.
* **Holdout 2020+.** No basket passes screening, so there is no out-of-sample result for the main experiment. Over the same period 22 of 24 candidate pairs survive in the pairs branch, which points to dimensionality (a VAR with $k^2P$ parameters on short formation windows) as the binding constraint, although the inherited thresholds are a confounding factor.
* **2008–2009 crisis.** One bond basket survives. `static` has the highest Sharpe; `full` trades once, so its near-zero drawdown reflects the absence of market participation rather than risk management over a meaningful number of trades. With $n = 1$ no inference is possible.

The multivariate branch should therefore be read as a proof of concept: the regime-aware framework improves on its static baseline in-sample, but its out-of-sample generalization remains untested with this universe. The weights are fixed after formation in all three specifications; a dynamic (multivariate Kalman) version of the weights is not implemented here.